سلول 1

مسیرها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(".")

DAY16_DIR = ROOT / "Data_ml/graph_dataset/day16_novel_predictions"

SCRNA_DIR = ROOT / "Data_raw/TISCH2"

OUTDIR = ROOT / "Results/day18_cancer_analysis"

OUTDIR.mkdir(parents=True, exist_ok=True)

سلول 2

لود Novel Predictionهای روز 16

In [ ]:
novel = pd.read_csv(
    DAY16_DIR / "novel_candidate_space_hgt_scored.csv"
)

print(novel.shape)
novel.head()

بعد چک کن ستون‌ها چیست:

In [ ]:
print(novel.columns.tolist())

Cell 3 — مسیر درست TISCH2 و پیدا کردن فایل‌ها

اول ببین فایل‌های single-cell دقیقاً کجا هستند:



اول این Debug را اجرا کن

Cell 6.1 — بررسی ساختار واقعی فایل‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob

PROJECT_ROOT = Path(".")

DAY16_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset" / "day16_novel_predictions"
DAY18_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset" / "day18_cancer_specific_analysis"
DAY18_DIR.mkdir(parents=True, exist_ok=True)

novel = pd.read_csv(
    DAY16_DIR / "top500_novel_predictions_hgt.csv"
)

print("novel:", novel.shape)
display(novel.head())

search_roots = [
    PROJECT_ROOT / "Data_raw",
    PROJECT_ROOT / "Data_interim",
    PROJECT_ROOT / "Data_proc",
]

scrna_hits = []

for root in search_roots:
    if not root.exists():
        continue
    
    for f in root.rglob("*"):
        name = f.name.lower()
        if (
            "crc" in name
            or "lihc" in name
            or "hcc" in name
            or "tisch" in name
        ):
            if f.is_file():
                scrna_hits.append(f)

scrna_hits = sorted(set(scrna_hits))

print("Found files:", len(scrna_hits))
for f in scrna_hits:
    print(f)

Cell 4 — نمایش ستون‌های فایل‌های CRC و LIHC

In [ ]:
for f in scrna_hits:
    print("\n" + "="*120)
    print(f)

    try:
        if f.suffix.lower() == ".csv":
            tmp = pd.read_csv(f, nrows=5)
        elif f.suffix.lower() in [".txt", ".tsv"]:
            tmp = pd.read_csv(f, sep="\t", nrows=5)
        elif f.suffix.lower() in [".xlsx", ".xls"]:
            tmp = pd.read_excel(f, nrows=5)
        else:
            print("unsupported:", f.suffix)
            continue

        print("shape preview:", tmp.shape)
        print("columns:", tmp.columns.tolist())
        display(tmp.head())

    except Exception as e:
        print("ERROR:", e)

Cell 5 — تابع عمومی برای خواندن expression matrixهای TISCH2

این نسخه چند حالت را پوشش می‌دهد:

* ستون اول gene باشد و بقیه cell type
* ستون‌ها gene باشند و ردیف‌ها cell type
* فایل majorlineage expression باشد

In [ ]:
def read_tisch_expression_file(path):
    path = Path(path)
    
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_csv(path, sep="\t")
    
    df = df.copy()
    
    # حذف ستون‌های کاملاً خالی
    df = df.dropna(axis=1, how="all")
    
    # اگر ستون اول gene-like بود
    first_col = df.columns[0]
    
    if first_col.lower() in ["gene", "genes", "symbol", "gene_symbol", "gene name", "gene_name"]:
        df = df.rename(columns={first_col: "gene"})
        df["gene"] = df["gene"].astype(str)
        expr = df.set_index("gene")
        expr = expr.apply(pd.to_numeric, errors="coerce")
        gene_scores = expr.mean(axis=1, skipna=True)
        return gene_scores
    
    # اگر ستون اول اسم gene نبود ولی احتمالاً geneها در index/ستون اول هستند
    if df.shape[1] > 2:
        possible_gene_col = df.columns[0]
        non_numeric_first = pd.to_numeric(df[possible_gene_col], errors="coerce").isna().mean()
        
        if non_numeric_first > 0.8:
            df = df.rename(columns={possible_gene_col: "gene"})
            df["gene"] = df["gene"].astype(str)
            expr = df.set_index("gene")
            expr = expr.apply(pd.to_numeric, errors="coerce")
            gene_scores = expr.mean(axis=1, skipna=True)
            return gene_scores
    
    # اگر geneها ستون هستند و ردیف‌ها cell type
    numeric_df = df.apply(pd.to_numeric, errors="coerce")
    
    if numeric_df.shape[1] > numeric_df.shape[0]:
        gene_scores = numeric_df.mean(axis=0, skipna=True)
        gene_scores.index = df.columns
        return gene_scores
    
    raise ValueError(f"Could not infer expression format for {path}")

Cell 6 — جدا کردن فایل‌های CRC و LIHC

In [ ]:
crc_files = [
    f for f in scrna_hits
    if "crc" in f.name.lower()
]

lihc_files = [
    f for f in scrna_hits
    if ("lihc" in f.name.lower() or "hcc" in f.name.lower())
]

print("CRC files:", len(crc_files))
for f in crc_files:
    print(f.name)

print("\nLIHC/HCC files:", len(lihc_files))
for f in lihc_files:
    print(f.name)

Cell 7 — ساخت meta-expression برای CRC و LIHC

In [ ]:
def build_meta_expression(files, label):
    all_scores = []

    for f in files:
        print("reading:", f.name)
        try:
            s = read_tisch_expression_file(f)
            s = s.dropna()
            s.name = f.stem
            all_scores.append(s)
            print("  genes:", len(s))
        except Exception as e:
            print("  ERROR:", e)

    if len(all_scores) == 0:
        raise ValueError(f"No valid expression files for {label}")

    mat = pd.concat(all_scores, axis=1)
    
    # میانگین بین datasetها
    meta = mat.mean(axis=1, skipna=True)
    meta = meta.dropna()
    
    # normalize 0-1
    meta_norm = (meta - meta.min()) / (meta.max() - meta.min())
    
    out = pd.DataFrame({
        "gene": meta_norm.index.astype(str),
        f"{label}_expr": meta_norm.values
    })
    
    out = out.groupby("gene", as_index=False)[f"{label}_expr"].max()
    
    return out, mat


crc_expr, crc_mat = build_meta_expression(crc_files, "crc")
lihc_expr, lihc_mat = build_meta_expression(lihc_files, "lihc")

print("crc_expr:", crc_expr.shape)
print("lihc_expr:", lihc_expr.shape)

display(crc_expr.head())
display(lihc_expr.head())

crc_expr.to_csv(DAY18_DIR / "crc_meta_expression.csv", index=False)
lihc_expr.to_csv(DAY18_DIR / "lihc_meta_expression.csv", index=False)

Cell 8 — اتصال expression به Novel Predictions

In [ ]:
cancer = novel.copy()

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "enz_gene", "crc_expr": "crc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "sub_gene", "crc_expr": "crc_sub_expr"}),
    on="sub_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "enz_gene", "lihc_expr": "lihc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "sub_gene", "lihc_expr": "lihc_sub_expr"}),
    on="sub_gene",
    how="left"
)

display(cancer.head())

print(cancer[[
    "crc_enz_expr",
    "crc_sub_expr",
    "lihc_enz_expr",
    "lihc_sub_expr"
]].isna().mean())

Cell 9 — محاسبه Cancer Activity

In [ ]:
def pair_activity(a, b):
    return np.sqrt(a * b)

cancer["crc_activity"] = pair_activity(
    cancer["crc_enz_expr"],
    cancer["crc_sub_expr"]
)

cancer["lihc_activity"] = pair_activity(
    cancer["lihc_enz_expr"],
    cancer["lihc_sub_expr"]
)

cancer["crc_score"] = (
    cancer["prob_hgt_mean"]
    *
    cancer["crc_activity"]
)

cancer["lihc_score"] = (
    cancer["prob_hgt_mean"]
    *
    cancer["lihc_activity"]
)

cancer["gi_score"] = (
    cancer[["crc_score", "lihc_score"]]
    .mean(axis=1, skipna=True)
)

display(
    cancer[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_activity",
            "lihc_activity",
            "crc_score",
            "lihc_score",
            "gi_score"
        ]
    ].head(20)
)

Cell 10 — Top CRC / LIHC / GI

In [ ]:
top_crc = (
    cancer
    .dropna(subset=["crc_score"])
    .sort_values("crc_score", ascending=False)
    .reset_index(drop=True)
)

top_lihc = (
    cancer
    .dropna(subset=["lihc_score"])
    .sort_values("lihc_score", ascending=False)
    .reset_index(drop=True)
)

top_gi = (
    cancer
    .dropna(subset=["gi_score"])
    .sort_values("gi_score", ascending=False)
    .reset_index(drop=True)
)

display(top_crc[["enzyme_class", "enz_gene", "sub_gene", "prob_hgt_mean", "crc_score"]].head(30))
display(top_lihc[["enzyme_class", "enz_gene", "sub_gene", "prob_hgt_mean", "lihc_score"]].head(30))
display(top_gi[["enzyme_class", "enz_gene", "sub_gene", "prob_hgt_mean", "gi_score"]].head(30))

top_crc.to_csv(DAY18_DIR / "top_crc_novel_interactions.csv", index=False)
top_lihc.to_csv(DAY18_DIR / "top_lihc_novel_interactions.csv", index=False)
top_gi.to_csv(DAY18_DIR / "top_gi_novel_interactions.csv", index=False)

Cell 11 — Shared و cancer-specific

In [ ]:
crc_top_set = set(top_crc.head(100)["pair_id"])
lihc_top_set = set(top_lihc.head(100)["pair_id"])

shared_ids = crc_top_set & lihc_top_set
crc_only_ids = crc_top_set - lihc_top_set
lihc_only_ids = lihc_top_set - crc_top_set

shared_gi = cancer[cancer["pair_id"].isin(shared_ids)].copy()
crc_only = cancer[cancer["pair_id"].isin(crc_only_ids)].copy()
lihc_only = cancer[cancer["pair_id"].isin(lihc_only_ids)].copy()

shared_gi = shared_gi.sort_values("gi_score", ascending=False)
crc_only = crc_only.sort_values("crc_score", ascending=False)
lihc_only = lihc_only.sort_values("lihc_score", ascending=False)

print("shared:", shared_gi.shape)
print("crc_only:", crc_only.shape)
print("lihc_only:", lihc_only.shape)

display(shared_gi[["enzyme_class", "enz_gene", "sub_gene", "crc_score", "lihc_score", "gi_score"]].head(30))
display(crc_only[["enzyme_class", "enz_gene", "sub_gene", "crc_score"]].head(30))
display(lihc_only[["enzyme_class", "enz_gene", "sub_gene", "lihc_score"]].head(30))

shared_gi.to_csv(DAY18_DIR / "shared_gi_top_interactions.csv", index=False)
crc_only.to_csv(DAY18_DIR / "crc_specific_top_interactions.csv", index=False)
lihc_only.to_csv(DAY18_DIR / "lihc_specific_top_interactions.csv", index=False)

Cell 12 — Enzyme ranking

In [ ]:
crc_enzyme_rank = (
    top_crc.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

lihc_enzyme_rank = (
    top_lihc.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

gi_enzyme_rank = (
    top_gi.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(crc_enzyme_rank.head(30))
display(lihc_enzyme_rank.head(30))
display(gi_enzyme_rank.head(30))

crc_enzyme_rank.to_csv(DAY18_DIR / "crc_enzyme_rank.csv", index=False)
lihc_enzyme_rank.to_csv(DAY18_DIR / "lihc_enzyme_rank.csv", index=False)
gi_enzyme_rank.to_csv(DAY18_DIR / "gi_enzyme_rank.csv", index=False)

Cell 13 — GI cancer pathway seed analysis

In [ ]:
gi_seed_genes = {
    "TP53",
    "CTNNB1",
    "AKT1",
    "YAP1",
    "WWTR1",
    "MAPK3",
    "MAPK8",
    "TNFAIP3",
    "EGFR",
    "MYC",
    "SMAD4",
    "APC",
    "KRAS",
    "PIK3CA",
    "BRAF",
    "ERBB2",
    "CDKN1A",
    "CCND1",
    "MTOR",
    "RPS6KB1",
}

cancer["contains_gi_seed"] = (
    cancer["enz_gene"].isin(gi_seed_genes)
    |
    cancer["sub_gene"].isin(gi_seed_genes)
)

top_crc["contains_gi_seed"] = top_crc["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

top_lihc["contains_gi_seed"] = top_lihc["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

top_gi["contains_gi_seed"] = top_gi["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

summary_seed = pd.DataFrame([
    {
        "set": "CRC_top100",
        "seed_rate": top_crc.head(100)["contains_gi_seed"].mean() * 100,
    },
    {
        "set": "LIHC_top100",
        "seed_rate": top_lihc.head(100)["contains_gi_seed"].mean() * 100,
    },
    {
        "set": "GI_top100",
        "seed_rate": top_gi.head(100)["contains_gi_seed"].mean() * 100,
    },
])

display(summary_seed)

summary_seed.to_csv(
    DAY18_DIR / "gi_seed_gene_enrichment_summary.csv",
    index=False
)

اول این Debug را اجرا کن

Cell 6.1 — بررسی ساختار واقعی فایل‌ها


In [ ]:
test_files = [
    crc_files[0],
    crc_files[1],
    lihc_files[0],
    lihc_files[1],
]

for f in test_files:
    print("\n" + "="*120)
    print(f)

    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f)
    else:
        df = pd.read_csv(f, sep="\t")

    print("shape:", df.shape)
    print("columns:", df.columns[:20].tolist())
    display(df.head())

Cell 6 را جایگزین کن — فقط فایل‌های majorlineage و بدون duplicate/zip

In [ ]:
crc_files = sorted(set([
    f for f in scrna_hits
    if "crc" in f.name.lower()
    and "majorlineage" in f.name.lower()
    and f.suffix.lower() in [".csv", ".txt", ".tsv"]
]))

lihc_files = sorted(set([
    f for f in scrna_hits
    if ("lihc" in f.name.lower() or "hcc" in f.name.lower())
    and "majorlineage" in f.name.lower()
    and f.suffix.lower() in [".csv", ".txt", ".tsv"]
]))

print("CRC majorlineage files:", len(crc_files))
for f in crc_files:
    print(f)

print("\nLIHC majorlineage files:", len(lihc_files))
for f in lihc_files:
    print(f)

Cell 7 را کامل جایگزین کن — Parser درست TISCH2

In [ ]:
def read_tisch_expression_file_fixed(path):
    path = Path(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_csv(path, sep="\t")

    df = df.dropna(axis=1, how="all").copy()

    # حالت رایج TISCH2:
    # ردیف‌ها = cell type
    # ستون‌ها = gene
    # ستون اول = cell type / cluster / lineage
    first_col = df.columns[0]

    first_col_non_numeric = pd.to_numeric(
        df[first_col],
        errors="coerce"
    ).isna().mean()

    if first_col_non_numeric > 0.8:
        expr = df.drop(columns=[first_col]).copy()
    else:
        expr = df.copy()

    expr = expr.apply(pd.to_numeric, errors="coerce")

    # میانگین expression هر ژن بین cell typeها
    gene_scores = expr.mean(axis=0, skipna=True)
    gene_scores = gene_scores.dropna()

    gene_scores.index = gene_scores.index.astype(str)

    # حذف ستون‌های غیرژن
    bad_cols = {
        "celltype", "cell_type", "cluster", "malignancy",
        "majorlineage", "minorlineage", "sample", "dataset"
    }

    gene_scores = gene_scores[
        ~gene_scores.index.str.lower().isin(bad_cols)
    ]

    return gene_scores


def build_meta_expression_fixed(files, label):
    all_scores = []

    for f in files:
        print("reading:", f.name)

        try:
            s = read_tisch_expression_file_fixed(f)
            s.name = f.stem
            all_scores.append(s)
            print("  genes:", len(s))

        except Exception as e:
            print("  ERROR:", e)

    if len(all_scores) == 0:
        raise ValueError(f"No valid expression files for {label}")

    mat = pd.concat(all_scores, axis=1)

    meta = mat.mean(axis=1, skipna=True)
    meta = meta.dropna()

    # log1p برای کم‌کردن اثر ژن‌های خیلی بالا
    meta = np.log1p(meta)

    # normalize 0-1
    if meta.max() > meta.min():
        meta_norm = (meta - meta.min()) / (meta.max() - meta.min())
    else:
        meta_norm = meta * 0

    out = pd.DataFrame({
        "gene": meta_norm.index.astype(str),
        f"{label}_expr": meta_norm.values
    })

    out = out.groupby("gene", as_index=False)[f"{label}_expr"].max()

    return out, mat


crc_expr, crc_mat = build_meta_expression_fixed(crc_files, "crc")
lihc_expr, lihc_mat = build_meta_expression_fixed(lihc_files, "lihc")

print("crc_expr:", crc_expr.shape)
print("lihc_expr:", lihc_expr.shape)

display(crc_expr.head(20))
display(lihc_expr.head(20))

crc_expr.to_csv(DAY18_DIR / "crc_meta_expression_fixed.csv", index=False)
lihc_expr.to_csv(DAY18_DIR / "lihc_meta_expression_fixed.csv", index=False)

بعد Cell 8 را دوباره اجرا کن

اگر درست شده باشد، این بخش دیگر نباید ۱۰۰٪ NaN بدهد:

Cell 8 — اتصال expression به Novel Predictions

In [ ]:
cancer = novel.copy()

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "enz_gene", "crc_expr": "crc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "sub_gene", "crc_expr": "crc_sub_expr"}),
    on="sub_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "enz_gene", "lihc_expr": "lihc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "sub_gene", "lihc_expr": "lihc_sub_expr"}),
    on="sub_gene",
    how="left"
)

display(cancer.head())

print(cancer[[
    "crc_enz_expr",
    "crc_sub_expr",
    "lihc_enz_expr",
    "lihc_sub_expr"
]].isna().mean())

قبل از اجرای Cell 7 این را اجرا کن

ببینیم واقعاً چند ژن تکراری داریم:

In [ ]:
for f in crc_files[:3]:

    print("\n", f.name)

    df = pd.read_csv(
        f,
        sep="\t"
    )

    gene_col = df.columns[0]

    dup = (
        df[gene_col]
        .value_counts()
        .loc[lambda x: x > 1]
    )

    print("duplicated genes:", len(dup))

    if len(dup):
        print(dup.head())

Cell 7 اصلاح‌شده

In [ ]:
def read_tisch_expression_file_gene_rows(path):

    path = Path(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_csv(path, sep="\t")

    df = df.dropna(axis=1, how="all").copy()

    gene_col = df.columns[0]

    df[gene_col] = df[gene_col].astype(str)

    expr = df.set_index(gene_col)

    expr = expr.apply(
        pd.to_numeric,
        errors="coerce"
    )

    gene_scores = expr.mean(
        axis=1,
        skipna=True
    )

    gene_scores = gene_scores.dropna()

    # FIX
    gene_scores = (
        gene_scores
        .groupby(level=0)
        .mean()
    )

    return gene_scores


def build_meta_expression_gene_rows(files, label):
    all_scores = []

    for f in sorted(set(files)):
        print("reading:", f.name)

        try:
            s = read_tisch_expression_file_gene_rows(f)
            s.name = f.stem
            all_scores.append(s)
            print("  genes:", len(s))

        except Exception as e:
            print("  ERROR:", e)

    if len(all_scores) == 0:
        raise ValueError(f"No valid expression files for {label}")

    mat = pd.concat(all_scores, axis=1)

    meta = mat.mean(axis=1, skipna=True)
    meta = meta.dropna()

    meta = np.log1p(meta)

    if meta.max() > meta.min():
        meta_norm = (meta - meta.min()) / (meta.max() - meta.min())
    else:
        meta_norm = meta * 0

    out = pd.DataFrame({
        "gene": meta_norm.index.astype(str),
        f"{label}_expr": meta_norm.values
    })

    out = out.groupby("gene", as_index=False)[f"{label}_expr"].max()

    return out, mat


crc_expr, crc_mat = build_meta_expression_gene_rows(crc_files, "crc")
lihc_expr, lihc_mat = build_meta_expression_gene_rows(lihc_files, "lihc")

print("crc_expr:", crc_expr.shape)
print("lihc_expr:", lihc_expr.shape)

display(crc_expr.head(20))
display(lihc_expr.head(20))

crc_expr.to_csv(DAY18_DIR / "crc_meta_expression_fixed.csv", index=False)
lihc_expr.to_csv(DAY18_DIR / "lihc_meta_expression_fixed.csv", index=False)

در فایل‌ها ستون اول ژن نیست؛ ستون اول خودش یک cell type مثل Malignant است. یعنی gene name احتمالاً در index فایل یا در ستون بدون اسم افتاده و با pd.read_csv درست خوانده نشده.

این Debug را اجرا کن تا ساختار واقعی فایل معلوم شود:

In [ ]:
f = crc_files[0]

print(f)

df1 = pd.read_csv(f, sep="\t")
print("normal read")
print(df1.shape)
print(df1.columns[:10].tolist())
display(df1.head())

df2 = pd.read_csv(f, sep="\t", index_col=0)
print("index_col=0 read")
print(df2.shape)
print(df2.columns[:10].tolist())
print(df2.index[:10].tolist())
display(df2.head())

df3 = pd.read_csv(f, sep="\t", header=None, nrows=5)
print("header=None read")
print(df3.shape)
display(df3)

Cell 7 نهایی و درست

کل Cell 7 قبلی را حذف کن و این را بگذار:

In [ ]:
def read_tisch_expression_file_final(path):
    path = Path(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, index_col=0)
    else:
        df = pd.read_csv(path, sep="\t", index_col=0)

    df = df.dropna(axis=1, how="all").copy()

    # index = gene names
    df.index = df.index.astype(str)

    expr = df.apply(pd.to_numeric, errors="coerce")

    # حذف indexهای عددی/مشکوک اگر وجود داشتند
    expr = expr[
        ~expr.index.str.fullmatch(r"\d+(\.\d+)?")
    ]

    gene_scores = expr.mean(axis=1, skipna=True)
    gene_scores = gene_scores.dropna()

    # اگر ژن تکراری بود، میانگین بگیر
    gene_scores = gene_scores.groupby(level=0).mean()

    return gene_scores


def build_meta_expression_final(files, label):
    all_scores = []

    for f in sorted(set(files)):
        print("reading:", f.name)

        try:
            s = read_tisch_expression_file_final(f)
            s.name = f.stem
            all_scores.append(s)
            print("  genes:", len(s))

        except Exception as e:
            print("  ERROR:", e)

    if len(all_scores) == 0:
        raise ValueError(f"No valid expression files for {label}")

    mat = pd.concat(all_scores, axis=1, join="outer")

    mat = mat.groupby(mat.index).mean()

    meta = mat.mean(axis=1, skipna=True)
    meta = meta.dropna()

    meta = np.log1p(meta)

    if meta.max() > meta.min():
        meta_norm = (meta - meta.min()) / (meta.max() - meta.min())
    else:
        meta_norm = meta * 0

    out = pd.DataFrame({
        "gene": meta_norm.index.astype(str),
        f"{label}_expr": meta_norm.values
    })

    out = out.groupby("gene", as_index=False)[f"{label}_expr"].max()

    return out, mat


crc_expr, crc_mat = build_meta_expression_final(crc_files, "crc")
lihc_expr, lihc_mat = build_meta_expression_final(lihc_files, "lihc")

print("crc_expr:", crc_expr.shape)
print("lihc_expr:", lihc_expr.shape)

display(crc_expr.head(20))
display(lihc_expr.head(20))

crc_expr.to_csv(DAY18_DIR / "crc_meta_expression_final.csv", index=False)
lihc_expr.to_csv(DAY18_DIR / "lihc_meta_expression_final.csv", index=False)

بعد Cell 8 را دوباره کامل اجرا کن

In [ ]:
cancer = novel.copy()

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "enz_gene", "crc_expr": "crc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    crc_expr.rename(columns={"gene": "sub_gene", "crc_expr": "crc_sub_expr"}),
    on="sub_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "enz_gene", "lihc_expr": "lihc_enz_expr"}),
    on="enz_gene",
    how="left"
)

cancer = cancer.merge(
    lihc_expr.rename(columns={"gene": "sub_gene", "lihc_expr": "lihc_sub_expr"}),
    on="sub_gene",
    how="left"
)

display(cancer.head())

print(cancer[
    [
        "crc_enz_expr",
        "crc_sub_expr",
        "lihc_enz_expr",
        "lihc_sub_expr"
    ]
].isna().mean())

Cell 9 — محاسبه Cancer Activity

In [ ]:
def pair_activity(a, b):
    return np.sqrt(a * b)

cancer["crc_activity"] = pair_activity(
    cancer["crc_enz_expr"],
    cancer["crc_sub_expr"]
)

cancer["lihc_activity"] = pair_activity(
    cancer["lihc_enz_expr"],
    cancer["lihc_sub_expr"]
)

cancer["crc_score"] = (
    cancer["prob_hgt_mean"]
    *
    cancer["crc_activity"]
)

cancer["lihc_score"] = (
    cancer["prob_hgt_mean"]
    *
    cancer["lihc_activity"]
)

cancer["gi_score"] = (
    cancer[["crc_score", "lihc_score"]]
    .mean(axis=1, skipna=True)
)

display(
    cancer[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_activity",
            "lihc_activity",
            "crc_score",
            "lihc_score",
            "gi_score"
        ]
    ].head(20)
)

Cell 10 — Top CRC / LIHC / GI

In [ ]:
top_crc = (
    cancer
    .dropna(subset=["crc_score"])
    .sort_values("crc_score", ascending=False)
    .reset_index(drop=True)
)

top_lihc = (
    cancer
    .dropna(subset=["lihc_score"])
    .sort_values("lihc_score", ascending=False)
    .reset_index(drop=True)
)

top_gi = (
    cancer
    .dropna(subset=["gi_score"])
    .sort_values("gi_score", ascending=False)
    .reset_index(drop=True)
)

display(
    top_crc[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_activity",
            "crc_score"
        ]
    ].head(30)
)

display(
    top_lihc[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "lihc_activity",
            "lihc_score"
        ]
    ].head(30)
)

display(
    top_gi[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_score",
            "lihc_score",
            "gi_score"
        ]
    ].head(30)
)

top_crc.to_csv(DAY18_DIR / "top_crc_novel_interactions.csv", index=False)
top_lihc.to_csv(DAY18_DIR / "top_lihc_novel_interactions.csv", index=False)
top_gi.to_csv(DAY18_DIR / "top_gi_novel_interactions.csv", index=False)

Cell 11 — Shared و cancer-specific

In [ ]:
crc_top_set = set(top_crc.head(100)["pair_id"])
lihc_top_set = set(top_lihc.head(100)["pair_id"])

shared_ids = crc_top_set & lihc_top_set
crc_only_ids = crc_top_set - lihc_top_set
lihc_only_ids = lihc_top_set - crc_top_set

shared_gi = cancer[cancer["pair_id"].isin(shared_ids)].copy()
crc_only = cancer[cancer["pair_id"].isin(crc_only_ids)].copy()
lihc_only = cancer[cancer["pair_id"].isin(lihc_only_ids)].copy()

shared_gi = shared_gi.sort_values("gi_score", ascending=False)
crc_only = crc_only.sort_values("crc_score", ascending=False)
lihc_only = lihc_only.sort_values("lihc_score", ascending=False)

print("shared:", shared_gi.shape)
print("crc_only:", crc_only.shape)
print("lihc_only:", lihc_only.shape)

display(
    shared_gi[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "crc_score",
            "lihc_score",
            "gi_score"
        ]
    ].head(30)
)

display(
    crc_only[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "crc_score"
        ]
    ].head(30)
)

display(
    lihc_only[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "lihc_score"
        ]
    ].head(30)
)

shared_gi.to_csv(DAY18_DIR / "shared_gi_top_interactions.csv", index=False)
crc_only.to_csv(DAY18_DIR / "crc_specific_top_interactions.csv", index=False)
lihc_only.to_csv(DAY18_DIR / "lihc_specific_top_interactions.csv", index=False)

Cell 12 — Enzyme ranking

In [ ]:
crc_enzyme_rank = (
    top_crc.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

lihc_enzyme_rank = (
    top_lihc.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

gi_enzyme_rank = (
    top_gi.head(200)
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(crc_enzyme_rank.head(30))
display(lihc_enzyme_rank.head(30))
display(gi_enzyme_rank.head(30))

crc_enzyme_rank.to_csv(DAY18_DIR / "crc_enzyme_rank.csv", index=False)
lihc_enzyme_rank.to_csv(DAY18_DIR / "lihc_enzyme_rank.csv", index=False)
gi_enzyme_rank.to_csv(DAY18_DIR / "gi_enzyme_rank.csv", index=False)

Cell 13 — GI seed gene enrichment

In [ ]:
gi_seed_genes = {
    "TP53",
    "CTNNB1",
    "AKT1",
    "YAP1",
    "WWTR1",
    "MAPK3",
    "MAPK8",
    "TNFAIP3",
    "EGFR",
    "MYC",
    "SMAD4",
    "APC",
    "KRAS",
    "PIK3CA",
    "BRAF",
    "ERBB2",
    "CDKN1A",
    "CCND1",
    "MTOR",
    "RPS6KB1",
}

cancer["contains_gi_seed"] = (
    cancer["enz_gene"].isin(gi_seed_genes)
    |
    cancer["sub_gene"].isin(gi_seed_genes)
)

top_crc["contains_gi_seed"] = top_crc["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

top_lihc["contains_gi_seed"] = top_lihc["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

top_gi["contains_gi_seed"] = top_gi["pair_id"].isin(
    cancer[cancer["contains_gi_seed"]]["pair_id"]
)

summary_seed = pd.DataFrame([
    {
        "set": "CRC_top100",
        "seed_rate": top_crc.head(100)["contains_gi_seed"].mean() * 100,
    },
    {
        "set": "LIHC_top100",
        "seed_rate": top_lihc.head(100)["contains_gi_seed"].mean() * 100,
    },
    {
        "set": "GI_top100",
        "seed_rate": top_gi.head(100)["contains_gi_seed"].mean() * 100,
    },
])

display(summary_seed)

summary_seed.to_csv(
    DAY18_DIR / "gi_seed_gene_enrichment_summary.csv",
    index=False
)

Cancer-driver filtering

Cell 14 — تعریف cancer-driver / pathway gene sets

In [ ]:
driver_gene_sets = {
    "p53_pathway": {
        "TP53", "MDM2", "CDKN1A", "BAX", "BBC3", "GADD45A"
    },
    "wnt_beta_catenin": {
        "CTNNB1", "APC", "AXIN1", "AXIN2", "GSK3B", "TNKS", "TNKS2"
    },
    "pi3k_akt_mtor": {
        "AKT1", "AKT2", "AKT3", "PIK3CA", "PTEN", "MTOR", "RICTOR", "RPS6KB1"
    },
    "mapk": {
        "MAPK1", "MAPK3", "MAPK8", "KRAS", "NRAS", "BRAF", "RAF1", "JUN", "FOS"
    },
    "nfkb_inflammation": {
        "TNFAIP3", "TRAF2", "TRAF6", "NFKB1", "RELA", "IKBKG", "RIPK1", "RIPK2"
    },
    "hippo": {
        "YAP1", "WWTR1", "LATS1", "LATS2", "TEAD1", "TEAD4"
    },
    "stress_autophagy": {
        "SQSTM1", "BECN1", "ATG5", "ATG7", "HSP90AA1", "ATF4", "XBP1"
    },
    "cell_cycle": {
        "CDC20", "CCNB1", "CCND1", "CDK1", "CDK2", "CHEK1", "AURKA", "BUB1"
    },
}

driver_genes = set()

for genes in driver_gene_sets.values():
    driver_genes |= genes

print("n driver genes:", len(driver_genes))
print(sorted(driver_genes))

Cell 15 — اضافه کردن برچسب pathway به هر interaction

In [ ]:
def assign_pathways(row):
    genes = {
        str(row["enz_gene"]),
        str(row["sub_gene"])
    }

    hits = []

    for pathway, gene_set in driver_gene_sets.items():
        if len(genes & gene_set) > 0:
            hits.append(pathway)

    return ";".join(hits)


cancer["driver_hit"] = (
    cancer["enz_gene"].isin(driver_genes)
    |
    cancer["sub_gene"].isin(driver_genes)
)

cancer["driver_pathways"] = cancer.apply(assign_pathways, axis=1)

top_crc["driver_hit"] = top_crc["pair_id"].isin(
    cancer[cancer["driver_hit"]]["pair_id"]
)

top_lihc["driver_hit"] = top_lihc["pair_id"].isin(
    cancer[cancer["driver_hit"]]["pair_id"]
)

top_gi["driver_hit"] = top_gi["pair_id"].isin(
    cancer[cancer["driver_hit"]]["pair_id"]
)

top_crc = top_crc.merge(
    cancer[["pair_id", "driver_pathways"]],
    on="pair_id",
    how="left"
)

top_lihc = top_lihc.merge(
    cancer[["pair_id", "driver_pathways"]],
    on="pair_id",
    how="left"
)

top_gi = top_gi.merge(
    cancer[["pair_id", "driver_pathways"]],
    on="pair_id",
    how="left"
)

display(top_gi.head(20))

Cell 16 — فیلتر driver candidates

In [ ]:
crc_driver = (
    top_crc[top_crc["driver_hit"]]
    .copy()
    .sort_values("crc_score", ascending=False)
    .reset_index(drop=True)
)

lihc_driver = (
    top_lihc[top_lihc["driver_hit"]]
    .copy()
    .sort_values("lihc_score", ascending=False)
    .reset_index(drop=True)
)

gi_driver = (
    top_gi[top_gi["driver_hit"]]
    .copy()
    .sort_values("gi_score", ascending=False)
    .reset_index(drop=True)
)

display(
    crc_driver[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_score",
            "driver_pathways"
        ]
    ].head(30)
)

display(
    lihc_driver[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "lihc_score",
            "driver_pathways"
        ]
    ].head(30)
)

display(
    gi_driver[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "gi_score",
            "driver_pathways"
        ]
    ].head(30)
)

crc_driver.to_csv(DAY18_DIR / "crc_driver_filtered_interactions.csv", index=False)
lihc_driver.to_csv(DAY18_DIR / "lihc_driver_filtered_interactions.csv", index=False)
gi_driver.to_csv(DAY18_DIR / "gi_driver_filtered_interactions.csv", index=False)

Cell 17 — Driver enrichment بعد از فیلتر

In [ ]:
driver_summary = pd.DataFrame([
    {
        "set": "CRC_top100",
        "driver_rate": top_crc.head(100)["driver_hit"].mean() * 100,
        "n_driver": int(top_crc.head(100)["driver_hit"].sum()),
    },
    {
        "set": "LIHC_top100",
        "driver_rate": top_lihc.head(100)["driver_hit"].mean() * 100,
        "n_driver": int(top_lihc.head(100)["driver_hit"].sum()),
    },
    {
        "set": "GI_top100",
        "driver_rate": top_gi.head(100)["driver_hit"].mean() * 100,
        "n_driver": int(top_gi.head(100)["driver_hit"].sum()),
    },
    {
        "set": "CRC_top500",
        "driver_rate": top_crc.head(500)["driver_hit"].mean() * 100,
        "n_driver": int(top_crc.head(500)["driver_hit"].sum()),
    },
    {
        "set": "LIHC_top500",
        "driver_rate": top_lihc.head(500)["driver_hit"].mean() * 100,
        "n_driver": int(top_lihc.head(500)["driver_hit"].sum()),
    },
    {
        "set": "GI_top500",
        "driver_rate": top_gi.head(500)["driver_hit"].mean() * 100,
        "n_driver": int(top_gi.head(500)["driver_hit"].sum()),
    },
])

display(driver_summary)

driver_summary.to_csv(
    DAY18_DIR / "driver_enrichment_summary.csv",
    index=False
)

Cell 18 — pathway frequency در GI driver candidates

In [ ]:
pathway_rows = []

for _, r in gi_driver.iterrows():
    pathways = str(r["driver_pathways"]).split(";")

    for p in pathways:
        if p and p != "nan":
            pathway_rows.append({
                "pair_id": r["pair_id"],
                "enzyme_class": r["enzyme_class"],
                "enzyme": r["enz_gene"],
                "substrate": r["sub_gene"],
                "pathway": p,
                "gi_score": r["gi_score"],
            })

pathway_freq = (
    pd.DataFrame(pathway_rows)
    .groupby("pathway")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(pathway_freq)

pathway_freq.to_csv(
    DAY18_DIR / "gi_driver_pathway_frequency.csv",
    index=False
)

Cell 19 — حذف AURKA از driver خروجی برای نسخه conservative

In [ ]:
suspicious_enzymes = {"AURKA"}

gi_driver_conservative = (
    gi_driver[
        ~gi_driver["enz_gene"].isin(suspicious_enzymes)
    ]
    .copy()
    .reset_index(drop=True)
)

crc_driver_conservative = (
    crc_driver[
        ~crc_driver["enz_gene"].isin(suspicious_enzymes)
    ]
    .copy()
    .reset_index(drop=True)
)

lihc_driver_conservative = (
    lihc_driver[
        ~lihc_driver["enz_gene"].isin(suspicious_enzymes)
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    gi_driver_conservative[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "gi_score",
            "driver_pathways"
        ]
    ].head(30)
)

gi_driver_conservative.to_csv(
    DAY18_DIR / "gi_driver_filtered_interactions_conservative_no_AURKA.csv",
    index=False
)

Cell 20 — Final Day18 priority table

In [ ]:
final_day18_priority = gi_driver_conservative.copy()

final_day18_priority["final_priority_score"] = (
    final_day18_priority["prob_hgt_mean"]
    *
    final_day18_priority["gi_score"]
    *
    (1 / (1 + final_day18_priority["prob_hgt_std"]))
)

final_day18_priority = (
    final_day18_priority
    .sort_values("final_priority_score", ascending=False)
    .reset_index(drop=True)
)

display(
    final_day18_priority[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "crc_score",
            "lihc_score",
            "gi_score",
            "driver_pathways",
            "final_priority_score"
        ]
    ].head(30)
)

final_day18_priority.to_csv(
    DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv",
    index=False
)